## Задание

- [x] Выбрать датасет
- [x] Определить задачу аппроксимации
- [x] Разбить датасет на обучающую и экспериментальную выборку
- [x] Провести корреляционный анализ
- [x] Выделить 1-2 переменные, которые влияют на выход
- [x] Разбить переменные на термы (распределить равномерно по графику)
- [x] Реализовать 4 функции принадлежности и для каждой построить графики
- [x] Реализовать машину нечёткого вывода (Синглтон / Мамдани / Такаги-Сугено)

## Данные

В качестве датасета возьмём данные об использовании Инстаграм\*: [Social Media User Analysis](https://www.kaggle.com/datasets/rockyt07/social-media-user-analysis/data).
<br/>Будем аппроксимировать шкалу воспринимаемого стресса (PSS-10 или perceived stress scale).

Разделим датасет на обучающую и экспериментальную выборку в соотношении 70 к 30.

*\* соцсеть Инстаграм признана экстремистской, её деятельность запрещена на территории РФ*

## Корреляционная матрица

Начнём с вычисления корреляции между параметрами. Выберем 3 наиболее коррелирующих с параметром PSS.

In [1]:
%use coroutines
%use dataframe
%use kandy

// read dataset
val rawData = DataFrame.read("data/instagram_usage_lifestyle.csv")
val learnData = rawData.head((rawData.rowsCount() * 0.7).toInt())
val experimentData = rawData.tail((rawData.rowsCount() * 0.3).toInt())
println("""
    Rows total: ${rawData.rowsCount()}
    Rows for learning: ${learnData.rowsCount()}
    Rows for experimental: ${experimentData.rowsCount()}
    """.trimIndent())

// evaluate correlation matrix (using pearson correlation coefficient)
val correlationMatrix = learnData.select { it.all() }.corr()
DISPLAY(correlationMatrix)

// select 3 most correlated params for PSS
val targetColumnName = "perceived_stress_score"
val correlatedColumns: List<String> =
    correlationMatrix.first {
        it["column"] == targetColumnName
    }.let { happinessRow ->
        (happinessRow.columnNames() - "column" - targetColumnName)
            .sortedBy { columnName ->
                (happinessRow[columnName] as Double).absoluteValue
            }.reversed().subList(0, 3)
    }

// show filtered correlation matrix
correlationMatrix.filter {
    it["column"] == targetColumnName
}.select("column", *correlatedColumns.toTypedArray())

Rows total: 1547896
Rows for learning: 1083527
Rows for experimental: 464368


column,user_id,age,exercise_hours_per_week,sleep_hours_per_night,perceived_stress_score,self_reported_happiness,body_mass_index,blood_pressure_systolic,blood_pressure_diastolic,daily_steps_count,weekly_work_hours,hobbies_count,social_events_per_month,books_read_per_year,volunteer_hours_per_month,travel_frequency_per_year,daily_active_minutes_instagram,sessions_per_day,posts_created_per_week,reels_watched_per_day,stories_viewed_per_day,likes_given_per_day,comments_written_per_day,dms_sent_per_week,dms_received_per_week,ads_viewed_per_day,ads_clicked_per_day,time_on_feed_per_day,time_on_explore_per_day,time_on_messages_per_day,time_on_reels_per_day,followers_count,following_count,notification_response_rate,account_creation_year,average_session_length_minutes,linked_accounts_count,user_engagement_score
user_id,"1,000000","-0,001529","-0,000848","0,000513","-0,000993","0,001430","0,000295","0,000217","-0,001184","-0,001283","0,000087","0,000758","0,000385","0,000885","0,000533","-0,000989","-0,001145","-0,001417","0,000405","-0,000464","-0,001399","-0,001084","-0,001318","-0,000643","-0,000812","-0,001573","-0,000874","-0,001157","-0,001203","-0,000565","-0,001605","0,001473","0,001286","0,000895","-0,001458","-0,000795","0,000191","0,001772"
age,"-0,001529","1,000000","-0,001245","0,001742","-0,000561","0,000046","0,001240","-0,000324","0,000152","0,000573","-0,000597","0,000163","0,000967","-0,002103","-0,002182","0,000716","-0,198355","-0,147162","-0,461694","-0,520696","-0,181949","-0,194439","-0,186522","-0,178327","-0,183164","-0,177253","-0,140733","-0,193724","-0,172226","-0,177797","-0,185592","-0,062281","-0,069518","0,000989","0,000790","-0,058114","0,001807","0,112297"
exercise_hours_per_week,"-0,000848","-0,001245","1,000000","0,001461","0,000083","-0,000308","0,000462","0,000224","-0,000585","0,000362","-0,001020","-0,001216","0,002176","-0,000633","-0,001474","-0,000873","0,000439","-0,000376","0,001080","0,000875","0,000758","0,000462","0,000467","-0,000426","0,000206","0,001247","0,000376","0,000211","-0,000373","0,000666","0,000289","-0,001275","-0,001169","0,000796","0,000028","0,002395","0,000032","-0,001907"
sleep_hours_per_night,"0,000513","0,001742","0,001461","1,000000","-0,000881","-0,000540","0,000425","0,001015","-0,000145","-0,000540","-0,000981","0,001001","-0,000736","-0,000042","-0,000837","0,000641","-0,000795","-0,000985","-0,000388","-0,001652","-0,000753","-0,000605","-0,000326","-0,000243","-0,000269","-0,000884","-0,001897","-0,000629","-0,001617","-0,001370","-0,000707","-0,000215","-0,000744","0,000902","-0,000998","0,000820","-0,000769","-0,000070"
perceived_stress_score,"-0,000993","-0,000561","0,000083","-0,000881","1,000000","0,000187","0,001441","0,000110","0,000201","-0,000076","0,000872","0,000300","-0,000128","-0,000257","-0,000445","-0,000293","0,834351","0,624127","0,376826","0,674719","0,813881","0,818492","0,787228","0,749826","0,769421","0,744556","0,592833","0,813425","0,723960","0,749131","0,779347","0,049720","0,057051","0,000005","-0,000110","0,159238","-0,000684","-0,435999"
self_reported_happiness,"0,001430","0,000046","-0,000308","-0,000540","0,000187","1,000000","0,000516","-0,000244","0,000368","-0,000017","-0,001976","-0,000502","0,000633","-0,000496","0,000202","0,000330","-0,372625","-0,277639","-0,133314","-0,295536","-0,344293","-0,365501","-0,351356","-0,334792","-0,343534","-0,331863","-0,264614","-0,363294","-0,323157","-0,334416","-0,348496","-0,017464","-0,019978","-0,000196","0,001166","-0,106418","-0,001203","0,268591"
body_mass_index,"0,000295","0,001240","0,000462","0,000425","0,001441","0,000516","1,000000","-0,000402","-0,001665","-0,001166","0,000001","0,000398","0,001312","-0,001720","-0,000765","0,001708","0,000690","-0,000115","0,001083","-0,000434","0,000927","0,000546","0,000604","0,000527","0,000474","0,000450","0,000022","0,001209","0,001095","0,000876","0,000764","0,000143","0,000614","0,000346","-0,000556","0,000376","-0,000881","-0,000085"
bloo

column,daily_active_minutes_instagram,likes_given_per_day,stories_viewed_per_day
perceived_stress_score,"0,834351","0,818492","0,813881"


Таким образом, `daily_active_minutes_instagram` (активное время в соцсети), `likes_given_per_day` (поставленные лайки) и `stories_viewed_per_day` (количество просмотренных reels) имеют сильную положительную корреляцию c PSS.
<br/>Визуализируем на графике точки и линейную регрессию:
$$\hat{\beta} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

In [2]:
import org.jetbrains.kotlinx.kandy.ir.Plot
import org.jetbrains.kotlinx.kandy.util.color.Color

// draw linear regression
/**
 * Columns must be [Int]
 */
fun displayLinearRegression(
    columnXName: String, columnXTitle: String,
    columnYName: String, columnYTitle: String,
    limit: Int = learnData.rowsCount()
) { // y = kx + b
    val x = learnData[columnXName].cast<Int>().toList()
    val y = learnData[columnYName].cast<Int>().toList()

    val meanX = x.average()
    val meanY = y.average()

    val k = x.zip(y).sumOf { (xi, yi) ->
        (xi - meanX) * (yi - meanY)
    } / x.sumOf {
        (it - meanX).pow(2)
    }
    val b = meanY - k * meanX

    val xLine = listOf(x.min().toDouble(), x.max().toDouble())
    val yLine = xLine.map { k * it + b }

    DISPLAY(learnData.head(limit).plot {
        points {
            x(column<Double>(columnXName)) {
                axis.name = columnXTitle
            }
            y(column<Int>(columnYName)) {
                axis.name = columnYTitle
            }
            color = Color.BLUE
        }
        line {
            x(xLine)
            y(yLine)
            color = Color.RED
        }
    })
}

displayLinearRegression(
    "daily_active_minutes_instagram", "Ежедневное использование (мин)",
    targetColumnName, "PSS",
    1000
)
displayLinearRegression(
    "likes_given_per_day", "Количество поставленных лайков (в день)",
    targetColumnName, "PSS",
    1000
)
displayLinearRegression(
    "stories_viewed_per_day", "Количество просмотренных reels",
    targetColumnName, "PSS",
    1000
)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="urdVfA"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"perceived_stress_score":[3.0,1.0,4.0,18.0,19.0,1.0,23.0,33.0,12.0,25.0,14.0,15.0,7.0,7.0,31.0,2.0,37.0,40.0,20.0,12.0,7.0,6.0,10.0,25.0,22.0,14.0,30.0,0.0,30.0,26.0,3.0,22.0,22.0,27.0,10.0,23.0,3.0,12.0,22.0,1.0,32.0,10.0,19.0,26.0,4.0,19.0,7.0,2.0,17.0,39.0,35.0,35.0,32.0,11.0,34.0,26.0,18.0,4.0,27.0,17.0,30.0,26.0,19.0,6.0,8.0,0.0,38.0,29.0,29.0,15.0,31.0,8.0,11.0,27.0,30.0,21.0,25.0,33.0,38.0,12.0,36.0,5.0,30.0,2.0,10.0,35.0,0.0,6.0,39.0,6.0,8.0,20.0,14.0,1.0,17.0,26.0,1.0,36.0,11.0,24.0,11.0,40.0,24.0,34.0,40.0,32.0,10.0,12.0,30.0,40.0,20.0,28.0,34.0,28.0,23.0,36.0,8.0,26.0,12.0,23.0,32.0,21.0,14.0,9.0,14.0,22.0,15.0,31.0,16.0,5.0,26.0,30.0,29.0,38.0,22.0,22.0,35.0,34.0,19.0,5.0,8.0,27.0,15.0,26.0,28.0,10.0,25.0,17.0,18.0,13.0,28.0,27.0,19.0,34.0,30.0,24.0,34.0,14.0,9.0,34.0,5.0,6.0,19.0,39.0,7.0,12.0,20.0,40.0,33.0,2.0,31.0,34.0,20.0,9.0,8.0,10.0,19.0,26.0,36.0,8.0,23.0,16.0,25.0,37.0,1.0,5.0,6.0,23.0,24.0,28.0,30.0,8.0,31.0,31.0,38.0,36.0,12.0,35.0,0.0,16.0,19.0,22.0,40.0,7.0,16.0,25.0,27.0,40.0,27.0,8.0,11.0,23.0,9.0,35.0,28.0,31.0,32.0,38.0,29.0,29.0,0.0,24.0,20.0,7.0,14.0,38.0,17.0,24.0,28.0,0.0,28.0,30.0,6.0,9.0,15.0,37.0,28.0,5.0,30.0,13.0,25.0,2.0,38.0,11.0,6.0,18.0,31.0,11.0,31.0,33.0,26.0,9.0,4.0,15.0,0.0,18.0,24.0,29.0,9.0,18.0,37.0,14.0,25.0,0.0,9.0,2.0,7.0,31.0,26.0,26.0,29.0,32.0,3.0,36.0,22.0,18.0,36.0,34.0,28.0,3.0,33.0,32.0,36.0,37.0,25.0,3.0,38.0,12.0,8.0,11.0,11.0,33.0,0.0,19.0,21.0,19.0,6.0,2.0,33.0,5.0,22.0,26.0,9.0,4.0,8.0,6.0,39.0,28.0,10.0,23.0,22.0,4.0,10.0,17.0,8.0,9.0,10.0,35.0,9.0,30.0,5.0,11.0,36.0,5.0,5.0,23.0,21.0,14.0,38.0,13.0,29.0,2.0,22.0,14.0,12.0,6.0,6.0,32.0,34.0,26.0,15.0,13.0,13.0,40.0,6.0,4.0,39.0,28.0,13.0,33.0,36.0,9.0,17.0,9.0,22.0,12.0,38.0,20.0,33.0,1.0,22.0,4.0,6.0,13.0,6.0,31.0,23.0,35.0,33.0,7.0,30.0,11.0,2.0,38.0,40.0,13.0,17.0,19.0,26.0,0.0,4.0,20.0,1.0,36.0,16.0,18.0,39.0,1.0,35.0,33.0,37.0,12.0,38.0,40.0,36.0,8.0,5.0,36.0,36.0,28.0,35.0,2.0,26.0,31.0,28.0,4.0,36.0,16.0,25.0,13.0,12.0,12.0,32.0,9.0,40.0,19.0,34.0,19.0,31.0,8.0,25.0,34.0,4.0,35.0,24.0,1.0,8.0,14.0,36.0,34.0,8.0,0.0,38.0,10.0,33.0,28.0,21.0,8.0,31.0,18.0,38.0,27.0,18.0,31.0,15.0,11.0,6.0,7.0,25.0,16.0,8.0,17.0,35.0,26.0,16.0,7.0,11.0,19.0,29.0,13.0,18.0,27.0,36.0,6.0,3.0,26.0,24.0,30.0,39.0,14.0,4.0,37.0,8.0,10.0,18.0,2.0,17.0,8.0,39.0,11.0,13.0,5.0,32.0,35.0,28.0,34.0,2.0,22.0,34.0,11.0,1.0,9.0,21.0,23.0,37.0,24.0,22.0,40.0,35.0,36.0,13.0,29.0,1.0,35.0,13.0,7.0,4.0,35.0,13.0,21.0,8.0,19.0,17.0,26.0,33.0,24.0,28.0,23.0,3.0,24.0,34.0,26.0,34.0,40.0,34.0,4.0,21.0,24.0,31.0,5.0,39.0,1.0,30.0,2.0,23.0,21.0,26.0,10.0,29.0,17.0,29.0,30.0,12.0,10.0,33.0,37.0,20.0,35.0,34.0,12.0,11.0,22.0,23.0,0.0,40.0,20.0,38.0,11.0,9.0,40.0,39.0,33.0,31.0,24.0,23.0,36.0,10.0,21.0,22.0,37.0,2.0,8.0,16.0,31.0,26.0,28.0,18.0,7.0,19.0,31.0,33.0,35.0,13.0,28.0,16.0,27.0,20.0,5.0,39.0,26.0,8.0,22.0,13.0,5.0,0.0,0.0,6.0,15.0,21.0,38.0,26.0,19.0,39.0,15.0,0.0,32.0,7.0,29.0,22.0,7.0,37.0,16.0,38.0,11.0,21.0,15.0,29.0,28.0,40.0,27.0,10.0,36.0,14.0,16.0,25.0,28.0,5.0,40.0,12.0,26.0,24.0,37.0,35.0,18.0,2.0,23.0,34.0,11.0,28.0,35.0,2.0,9.0,33.0,16.0,1.0,36.0,39.0,31.0,10.0,29.0,20.0,12.0,17.0,29.0,36.0,35.0,22.0,38.0,12.0,36.0,7.0,40.0,37.0,1.0,25.0,3.0,13.0,2.0,21.0,33.0,11.0,2.0,8.0,8.0,13.0,2.0,35.0,21.0,5.0,21.0,34.0,36.0,26.0,32.0,34.0,8.0,37.0,5.0,23.0,30.0,29.0,38.0,14.0,31.0,17.0,17.0,33.0,5.0,6.0,7.0,28.0,12.0,36.0,40.0,32.0,25.0,22.0,32.0,1.0,21.0,35.0,13.0,40.0,3.0,17.0,1.0,8.0,5.0,15.0,18.0,5.0,17.0,23.0,2.0,22.0,1.0,38.0,14.0,1.0,13.0,15.0,0.0,15.0,37.0,23.0,10.0,1.0,32.0,25.0,3.0,12.0,30.0,19.0,3

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="rO3OCb"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"perceived_stress_score":[3.0,1.0,4.0,18.0,19.0,1.0,23.0,33.0,12.0,25.0,14.0,15.0,7.0,7.0,31.0,2.0,37.0,40.0,20.0,12.0,7.0,6.0,10.0,25.0,22.0,14.0,30.0,0.0,30.0,26.0,3.0,22.0,22.0,27.0,10.0,23.0,3.0,12.0,22.0,1.0,32.0,10.0,19.0,26.0,4.0,19.0,7.0,2.0,17.0,39.0,35.0,35.0,32.0,11.0,34.0,26.0,18.0,4.0,27.0,17.0,30.0,26.0,19.0,6.0,8.0,0.0,38.0,29.0,29.0,15.0,31.0,8.0,11.0,27.0,30.0,21.0,25.0,33.0,38.0,12.0,36.0,5.0,30.0,2.0,10.0,35.0,0.0,6.0,39.0,6.0,8.0,20.0,14.0,1.0,17.0,26.0,1.0,36.0,11.0,24.0,11.0,40.0,24.0,34.0,40.0,32.0,10.0,12.0,30.0,40.0,20.0,28.0,34.0,28.0,23.0,36.0,8.0,26.0,12.0,23.0,32.0,21.0,14.0,9.0,14.0,22.0,15.0,31.0,16.0,5.0,26.0,30.0,29.0,38.0,22.0,22.0,35.0,34.0,19.0,5.0,8.0,27.0,15.0,26.0,28.0,10.0,25.0,17.0,18.0,13.0,28.0,27.0,19.0,34.0,30.0,24.0,34.0,14.0,9.0,34.0,5.0,6.0,19.0,39.0,7.0,12.0,20.0,40.0,33.0,2.0,31.0,34.0,20.0,9.0,8.0,10.0,19.0,26.0,36.0,8.0,23.0,16.0,25.0,37.0,1.0,5.0,6.0,23.0,24.0,28.0,30.0,8.0,31.0,31.0,38.0,36.0,12.0,35.0,0.0,16.0,19.0,22.0,40.0,7.0,16.0,25.0,27.0,40.0,27.0,8.0,11.0,23.0,9.0,35.0,28.0,31.0,32.0,38.0,29.0,29.0,0.0,24.0,20.0,7.0,14.0,38.0,17.0,24.0,28.0,0.0,28.0,30.0,6.0,9.0,15.0,37.0,28.0,5.0,30.0,13.0,25.0,2.0,38.0,11.0,6.0,18.0,31.0,11.0,31.0,33.0,26.0,9.0,4.0,15.0,0.0,18.0,24.0,29.0,9.0,18.0,37.0,14.0,25.0,0.0,9.0,2.0,7.0,31.0,26.0,26.0,29.0,32.0,3.0,36.0,22.0,18.0,36.0,34.0,28.0,3.0,33.0,32.0,36.0,37.0,25.0,3.0,38.0,12.0,8.0,11.0,11.0,33.0,0.0,19.0,21.0,19.0,6.0,2.0,33.0,5.0,22.0,26.0,9.0,4.0,8.0,6.0,39.0,28.0,10.0,23.0,22.0,4.0,10.0,17.0,8.0,9.0,10.0,35.0,9.0,30.0,5.0,11.0,36.0,5.0,5.0,23.0,21.0,14.0,38.0,13.0,29.0,2.0,22.0,14.0,12.0,6.0,6.0,32.0,34.0,26.0,15.0,13.0,13.0,40.0,6.0,4.0,39.0,28.0,13.0,33.0,36.0,9.0,17.0,9.0,22.0,12.0,38.0,20.0,33.0,1.0,22.0,4.0,6.0,13.0,6.0,31.0,23.0,35.0,33.0,7.0,30.0,11.0,2.0,38.0,40.0,13.0,17.0,19.0,26.0,0.0,4.0,20.0,1.0,36.0,16.0,18.0,39.0,1.0,35.0,33.0,37.0,12.0,38.0,40.0,36.0,8.0,5.0,36.0,36.0,28.0,35.0,2.0,26.0,31.0,28.0,4.0,36.0,16.0,25.0,13.0,12.0,12.0,32.0,9.0,40.0,19.0,34.0,19.0,31.0,8.0,25.0,34.0,4.0,35.0,24.0,1.0,8.0,14.0,36.0,34.0,8.0,0.0,38.0,10.0,33.0,28.0,21.0,8.0,31.0,18.0,38.0,27.0,18.0,31.0,15.0,11.0,6.0,7.0,25.0,16.0,8.0,17.0,35.0,26.0,16.0,7.0,11.0,19.0,29.0,13.0,18.0,27.0,36.0,6.0,3.0,26.0,24.0,30.0,39.0,14.0,4.0,37.0,8.0,10.0,18.0,2.0,17.0,8.0,39.0,11.0,13.0,5.0,32.0,35.0,28.0,34.0,2.0,22.0,34.0,11.0,1.0,9.0,21.0,23.0,37.0,24.0,22.0,40.0,35.0,36.0,13.0,29.0,1.0,35.0,13.0,7.0,4.0,35.0,13.0,21.0,8.0,19.0,17.0,26.0,33.0,24.0,28.0,23.0,3.0,24.0,34.0,26.0,34.0,40.0,34.0,4.0,21.0,24.0,31.0,5.0,39.0,1.0,30.0,2.0,23.0,21.0,26.0,10.0,29.0,17.0,29.0,30.0,12.0,10.0,33.0,37.0,20.0,35.0,34.0,12.0,11.0,22.0,23.0,0.0,40.0,20.0,38.0,11.0,9.0,40.0,39.0,33.0,31.0,24.0,23.0,36.0,10.0,21.0,22.0,37.0,2.0,8.0,16.0,31.0,26.0,28.0,18.0,7.0,19.0,31.0,33.0,35.0,13.0,28.0,16.0,27.0,20.0,5.0,39.0,26.0,8.0,22.0,13.0,5.0,0.0,0.0,6.0,15.0,21.0,38.0,26.0,19.0,39.0,15.0,0.0,32.0,7.0,29.0,22.0,7.0,37.0,16.0,38.0,11.0,21.0,15.0,29.0,28.0,40.0,27.0,10.0,36.0,14.0,16.0,25.0,28.0,5.0,40.0,12.0,26.0,24.0,37.0,35.0,18.0,2.0,23.0,34.0,11.0,28.0,35.0,2.0,9.0,33.0,16.0,1.0,36.0,39.0,31.0,10.0,29.0,20.0,12.0,17.0,29.0,36.0,35.0,22.0,38.0,12.0,36.0,7.0,40.0,37.0,1.0,25.0,3.0,13.0,2.0,21.0,33.0,11.0,2.0,8.0,8.0,13.0,2.0,35.0,21.0,5.0,21.0,34.0,36.0,26.0,32.0,34.0,8.0,37.0,5.0,23.0,30.0,29.0,38.0,14.0,31.0,17.0,17.0,33.0,5.0,6.0,7.0,28.0,12.0,36.0,40.0,32.0,25.0,22.0,32.0,1.0,21.0,35.0,13.0,40.0,3.0,17.0,1.0,8.0,5.0,15.0,18.0,5.0,17.0,23.0,2.0,22.0,1.0,38.0,14.0,1.0,13.0,15.0,0.0,15.0,37.0,23.0,10.0,1.0,32.0,25.0,3.0,12.0,30.0,19.0,3

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="V0WBrh"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"perceived_stress_score":[3.0,1.0,4.0,18.0,19.0,1.0,23.0,33.0,12.0,25.0,14.0,15.0,7.0,7.0,31.0,2.0,37.0,40.0,20.0,12.0,7.0,6.0,10.0,25.0,22.0,14.0,30.0,0.0,30.0,26.0,3.0,22.0,22.0,27.0,10.0,23.0,3.0,12.0,22.0,1.0,32.0,10.0,19.0,26.0,4.0,19.0,7.0,2.0,17.0,39.0,35.0,35.0,32.0,11.0,34.0,26.0,18.0,4.0,27.0,17.0,30.0,26.0,19.0,6.0,8.0,0.0,38.0,29.0,29.0,15.0,31.0,8.0,11.0,27.0,30.0,21.0,25.0,33.0,38.0,12.0,36.0,5.0,30.0,2.0,10.0,35.0,0.0,6.0,39.0,6.0,8.0,20.0,14.0,1.0,17.0,26.0,1.0,36.0,11.0,24.0,11.0,40.0,24.0,34.0,40.0,32.0,10.0,12.0,30.0,40.0,20.0,28.0,34.0,28.0,23.0,36.0,8.0,26.0,12.0,23.0,32.0,21.0,14.0,9.0,14.0,22.0,15.0,31.0,16.0,5.0,26.0,30.0,29.0,38.0,22.0,22.0,35.0,34.0,19.0,5.0,8.0,27.0,15.0,26.0,28.0,10.0,25.0,17.0,18.0,13.0,28.0,27.0,19.0,34.0,30.0,24.0,34.0,14.0,9.0,34.0,5.0,6.0,19.0,39.0,7.0,12.0,20.0,40.0,33.0,2.0,31.0,34.0,20.0,9.0,8.0,10.0,19.0,26.0,36.0,8.0,23.0,16.0,25.0,37.0,1.0,5.0,6.0,23.0,24.0,28.0,30.0,8.0,31.0,31.0,38.0,36.0,12.0,35.0,0.0,16.0,19.0,22.0,40.0,7.0,16.0,25.0,27.0,40.0,27.0,8.0,11.0,23.0,9.0,35.0,28.0,31.0,32.0,38.0,29.0,29.0,0.0,24.0,20.0,7.0,14.0,38.0,17.0,24.0,28.0,0.0,28.0,30.0,6.0,9.0,15.0,37.0,28.0,5.0,30.0,13.0,25.0,2.0,38.0,11.0,6.0,18.0,31.0,11.0,31.0,33.0,26.0,9.0,4.0,15.0,0.0,18.0,24.0,29.0,9.0,18.0,37.0,14.0,25.0,0.0,9.0,2.0,7.0,31.0,26.0,26.0,29.0,32.0,3.0,36.0,22.0,18.0,36.0,34.0,28.0,3.0,33.0,32.0,36.0,37.0,25.0,3.0,38.0,12.0,8.0,11.0,11.0,33.0,0.0,19.0,21.0,19.0,6.0,2.0,33.0,5.0,22.0,26.0,9.0,4.0,8.0,6.0,39.0,28.0,10.0,23.0,22.0,4.0,10.0,17.0,8.0,9.0,10.0,35.0,9.0,30.0,5.0,11.0,36.0,5.0,5.0,23.0,21.0,14.0,38.0,13.0,29.0,2.0,22.0,14.0,12.0,6.0,6.0,32.0,34.0,26.0,15.0,13.0,13.0,40.0,6.0,4.0,39.0,28.0,13.0,33.0,36.0,9.0,17.0,9.0,22.0,12.0,38.0,20.0,33.0,1.0,22.0,4.0,6.0,13.0,6.0,31.0,23.0,35.0,33.0,7.0,30.0,11.0,2.0,38.0,40.0,13.0,17.0,19.0,26.0,0.0,4.0,20.0,1.0,36.0,16.0,18.0,39.0,1.0,35.0,33.0,37.0,12.0,38.0,40.0,36.0,8.0,5.0,36.0,36.0,28.0,35.0,2.0,26.0,31.0,28.0,4.0,36.0,16.0,25.0,13.0,12.0,12.0,32.0,9.0,40.0,19.0,34.0,19.0,31.0,8.0,25.0,34.0,4.0,35.0,24.0,1.0,8.0,14.0,36.0,34.0,8.0,0.0,38.0,10.0,33.0,28.0,21.0,8.0,31.0,18.0,38.0,27.0,18.0,31.0,15.0,11.0,6.0,7.0,25.0,16.0,8.0,17.0,35.0,26.0,16.0,7.0,11.0,19.0,29.0,13.0,18.0,27.0,36.0,6.0,3.0,26.0,24.0,30.0,39.0,14.0,4.0,37.0,8.0,10.0,18.0,2.0,17.0,8.0,39.0,11.0,13.0,5.0,32.0,35.0,28.0,34.0,2.0,22.0,34.0,11.0,1.0,9.0,21.0,23.0,37.0,24.0,22.0,40.0,35.0,36.0,13.0,29.0,1.0,35.0,13.0,7.0,4.0,35.0,13.0,21.0,8.0,19.0,17.0,26.0,33.0,24.0,28.0,23.0,3.0,24.0,34.0,26.0,34.0,40.0,34.0,4.0,21.0,24.0,31.0,5.0,39.0,1.0,30.0,2.0,23.0,21.0,26.0,10.0,29.0,17.0,29.0,30.0,12.0,10.0,33.0,37.0,20.0,35.0,34.0,12.0,11.0,22.0,23.0,0.0,40.0,20.0,38.0,11.0,9.0,40.0,39.0,33.0,31.0,24.0,23.0,36.0,10.0,21.0,22.0,37.0,2.0,8.0,16.0,31.0,26.0,28.0,18.0,7.0,19.0,31.0,33.0,35.0,13.0,28.0,16.0,27.0,20.0,5.0,39.0,26.0,8.0,22.0,13.0,5.0,0.0,0.0,6.0,15.0,21.0,38.0,26.0,19.0,39.0,15.0,0.0,32.0,7.0,29.0,22.0,7.0,37.0,16.0,38.0,11.0,21.0,15.0,29.0,28.0,40.0,27.0,10.0,36.0,14.0,16.0,25.0,28.0,5.0,40.0,12.0,26.0,24.0,37.0,35.0,18.0,2.0,23.0,34.0,11.0,28.0,35.0,2.0,9.0,33.0,16.0,1.0,36.0,39.0,31.0,10.0,29.0,20.0,12.0,17.0,29.0,36.0,35.0,22.0,38.0,12.0,36.0,7.0,40.0,37.0,1.0,25.0,3.0,13.0,2.0,21.0,33.0,11.0,2.0,8.0,8.0,13.0,2.0,35.0,21.0,5.0,21.0,34.0,36.0,26.0,32.0,34.0,8.0,37.0,5.0,23.0,30.0,29.0,38.0,14.0,31.0,17.0,17.0,33.0,5.0,6.0,7.0,28.0,12.0,36.0,40.0,32.0,25.0,22.0,32.0,1.0,21.0,35.0,13.0,40.0,3.0,17.0,1.0,8.0,5.0,15.0,18.0,5.0,17.0,23.0,2.0,22.0,1.0,38.0,14.0,1.0,13.0,15.0,0.0,15.0,37.0,23.0,10.0,1.0,32.0,25.0,3.0,12.0,30.0,19.0,3

Распределим параметры равномерно по термам.
Ниже приведены формулы разных функций принадлежности, но мы будем использовать только треугольную:

**Треугольная**:
$$
% Треугольная (a, b, c — левая граница, вершина, правая граница)
\mu(x) = \begin{cases}
0, & x \leq a \\
\frac{x - a}{b - a}, & a < x \leq b \\
\frac{c - x}{c - b}, & b < x < c \\
0, & x \geq c
\end{cases}
$$

**Трапецеидальная**:
$$
% Трапецеидальная (a, b, c, d — границы и плато)
\mu(x) = \begin{cases}
0, & x \leq a \\
\frac{x - a}{b - a}, & a < x < b \\
1, & b \leq x \leq c \\
\frac{d - x}{d - c}, & c < x < d \\
0, & x \geq d
\end{cases}
$$

**Параболическая**:
$$
% Параболическая (a, b — границы)
\mu(x) = \begin{cases}
0, & x \leq a \\
1 - \left(\frac{x - b}{b - a}\right)^2, & a < x \leq b \\
1 - \left(\frac{x - b}{c - b}\right)^2, & b < x < c \\
0, & x \geq c
\end{cases}
$$

**Гаусса**:
$$
% Гауссова (c — центр, σ — ширина)
\mu(x) = e^{-\frac{(x - c)^2}{2\sigma^2}}
$$

In [3]:
/* API */

enum class MembershipFunctionType {
    TRIANGULAR,
    TRAPEZOIDAL,
    PARABOLIC,
    GAUSSIAN
}

data class LinguisticVariable(
    val columnName: String,
    val userFriendlyName: String,
    val membershipFunctionType: MembershipFunctionType,
    val termNames: List<String>
) {
    val minValue: Int
    val maxValue: Int

    init {
        learnData[columnName]
            .cast<Int>()
            .toList()
            .run {
                minValue = min()
                maxValue = max()
            }
    }
}

fun interface TermChartBuilder {
    /**
     * [fromX], [toX] - both inclusive
     */
    fun append(
        // input
        fromX: Double,
        toX: Double,
        termName: String,
        // output
        bufferX: MutableList<Double>,
        bufferY: MutableList<Double>,
        bufferTerm: MutableList<String>
    )
}

fun displayTerms(
    variable: LinguisticVariable,
    builder: TermChartBuilder
) {
    val step = (variable.maxValue - variable.minValue) / (variable.termNames.size.toDouble() - 1) * 2
    val fromX = variable.minValue - step / 2

    val bufferX = ArrayList<Double>()
    val bufferY = ArrayList<Double>()
    val bufferTerm = ArrayList<String>()

    variable.termNames.forEachIndexed { i, termName ->
        builder.append(
            fromX + step / 2 * i, fromX + step * (i / 2.0 + 1), termName,
            bufferX, bufferY, bufferTerm
        )
    }

    DISPLAY(learnData.plot {
        layout.title = variable.userFriendlyName
        x.axis.limits = variable.minValue.toDouble() ..  variable.maxValue.toDouble()
        y.axis.limits = 0 .. 1
        line {
            x(bufferX)
            y(bufferY)
            color(bufferTerm) { legend.name = "Терм" }
        }
    })
}

/* Term functions */

fun appendTriangularTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    bufferX += listOf(fromX, fromX + (toX - fromX) / 2.0, toX)
    bufferY += listOf(0.0, 1.0, 0.0)
    bufferTerm += List(3) { termName }
}

fun appendTrapezoidalTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    bufferX += listOf(
        fromX,
        fromX + (toX - fromX) / 3.0,
        fromX + (toX - fromX) / 3.0 * 2,
        toX
    )
    bufferY += listOf(0.0, 1.0, 1.0, 0.0)
    bufferTerm += List(4) { termName }
}

fun appendParabolicTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    val valuesX = (0 until 100).map { fromX + (toX - fromX) / 100 * it }
    val centerX = (toX - fromX) / 2 + fromX
    bufferX += valuesX
    bufferY += valuesX.map { 1 - ((it - centerX) / (centerX - fromX)).pow(2) }
    bufferTerm += List(100) { termName }
}

fun appendGaussianTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    val valuesX = (0 until 100).map { fromX + (toX - fromX) / 100 * it }
    val centerX = (toX - fromX) / 2 + fromX
    bufferX += valuesX
    bufferY += valuesX.map {
        val sigma = (centerX - fromX) / 3 // (centerX - fromX) is too big
        val numerator = - (it - centerX).pow(2)
        val denominator = 2 * sigma.pow(2)
        exp(numerator / denominator)
    }
    bufferTerm += List(100) { termName }
}

/* Impl */

val variables: List<LinguisticVariable> = listOf(
    LinguisticVariable(
        userFriendlyName = "PSS-10",
        columnName = "perceived_stress_score",
        termNames = listOf("низкий стресс", "умеренный стресс", "высокий стресс"),
        membershipFunctionType = MembershipFunctionType.TRIANGULAR
    ),

    LinguisticVariable(
        userFriendlyName = "Ежедневное использование соцсети (мин)",
        columnName = "daily_active_minutes_instagram",
        termNames = listOf("немного", "много", "очень много", "вообще не расстаётся с телефоном"),
        membershipFunctionType = MembershipFunctionType.TRIANGULAR
    ),

    LinguisticVariable(
        userFriendlyName = "Лайков поставлено (в день)",
        columnName = "likes_given_per_day",
        termNames = listOf("немного", "много", "очень много"),
        membershipFunctionType = MembershipFunctionType.TRIANGULAR
    ),

    LinguisticVariable(
        userFriendlyName = "Просмотрено reels",
        columnName = "stories_viewed_per_day",
        termNames = listOf("немного", "много", "очень много"),
        membershipFunctionType = MembershipFunctionType.TRIANGULAR
    )
)

variables // display membership functions charts
    .map {
        it to TermChartBuilder(
            when (it.membershipFunctionType) {
                MembershipFunctionType.TRIANGULAR -> ::appendTriangularTerm
                MembershipFunctionType.TRAPEZOIDAL -> ::appendTrapezoidalTerm
                MembershipFunctionType.PARABOLIC -> ::appendParabolicTerm
                MembershipFunctionType.GAUSSIAN -> ::appendGaussianTerm
            }
        )
    }.toMap()
    .forEach(::displayTerms)

fun dataFrameOf(headers: List<String>, cells: List<List<*>>): DataFrame<*> =
    dataFrameOf(headers)(*cells.flatMap { it }.toTypedArray())

DISPLAY( // display table of variables
    dataFrameOf(
        headers = listOf("Переменная", "Столбец", "Минимальное значение", "Максимальное значение"),
        cells = variables.map { listOf(it.userFriendlyName, it.columnName, it.minValue, it.maxValue) }
    )
)

/* Used only by following code blocks */

val inputVariables: List<LinguisticVariable> = variables.drop(1)
val outputVariable: LinguisticVariable = variables.first()

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="hu5DGi"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"PSS-10"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[0.0,40.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["низкий стресс","низкий стресс","низкий стресс","умеренный стресс","умеренный стресс","умеренный стресс","высокий стресс","высокий стресс","высокий стресс"],
"x":[-20.0,0.0,20.0,0.0,20.0,40.0,20.0,40.0,60.0],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"11"
};
 var containerDiv = document.getElementById("hu5DGi");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 PSS-10 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 низкий стресс 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 умеренный стресс 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 высокий стресс

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="09yHLV"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Ежедневное использование соцсети (мин)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[5.0,580.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["немного","немного","немного","много","много","много","очень много","очень много","очень много","вообще не расстаётся с телефоном","вообще не расстаётся с телефоном","вообще не расстаётся с телефоном"],
"x":[-186.66666666666666,5.0,196.66666666666666,5.0,196.66666666666669,388.33333333333337,196.66666666666666,388.33333333333337,580.0,388.33333333333337,580.0,771.6666666666666],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"14"
};
 var containerDiv = document.getElementById("09yHLV");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Ежедневное использование соцсети (мин) 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 немного 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 очень много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 вообще не расстаётся с 
 
 
 телефоном

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="YVpqJf"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Лайков поставлено (в день)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[8.0,350.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["немного","немного","немного","много","много","много","очень много","очень много","очень много"],
"x":[-163.0,8.0,179.0,8.0,179.0,350.0,179.0,350.0,521.0],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"17"
};
 var containerDiv = document.getElementById("YVpqJf");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 150 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 250 
 
 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 
 
 350 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Лайков поставлено (в день) 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 немного 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 очень много

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="AAvhUx"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Просмотрено reels"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[12.0,150.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["немного","немного","немного","много","много","много","очень много","очень много","очень много"],
"x":[-57.0,12.0,81.0,12.0,81.0,150.0,81.0,150.0,219.0],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"20"
};
 var containerDiv = document.getElementById("AAvhUx");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 120 
 
 
 
 
 
 
 
 
 140 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Просмотрено reels 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 немного 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 очень много

Переменная,Столбец,Минимальное значение,Максимальное значение
PSS-10,perceived_stress_score,0,40
Ежедневное использование соцсети (мин),daily_active_minutes_instagram,5,580
Лайков поставлено (в день),likes_given_per_day,8,350
Просмотрено reels,stories_viewed_per_day,12,150


Определим функции принадлежности, соответствующие графикам каждого терма, а также некоторый программный интерфейс над ними:

In [4]:
/* API */

fun interface MembershipFunction {
    operator fun invoke(x: Double): Double
}

infix fun MembershipFunction.fuzzyOr(function: MembershipFunction) = MembershipFunction { max(this(it), function(it)) }
infix fun MembershipFunction.fuzzyAnd(function: MembershipFunction) = MembershipFunction { min(this(it), function(it)) }

/* Membership functions */

fun interface MembershipFunctionBuilder {
    operator fun invoke(
        fromX: Double,
        toX: Double
    ): MembershipFunction
}

class TriangularMembershipFunction(
    private val fromX: Double,
    private val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val centerX = fromX + (toX - fromX) / 2.0
        return when {
            x <= fromX -> 0.0
            x <= centerX -> (x - fromX) / (centerX - fromX)
            x < toX -> (toX - x) / (toX - centerX)
            else -> 0.0 // x >= toX
        }
    }
}

class TrapezoidalMembershipFunction(
    private val fromX: Double,
    private val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val thirdX = fromX + (toX - fromX) / 3.0
        val twoThirdX = fromX + (toX - fromX) / 3.0 * 2
        return when {
            x <= fromX -> 0.0
            x < thirdX -> (x - fromX) / (thirdX - fromX)
            x <= twoThirdX -> 1.0
            x < toX -> (toX - x) / (toX - twoThirdX)
            else -> 0.0 // x >= toX
        }
    }
}

class ParabolicMembershipFunction(
    private val fromX: Double,
    private val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val centerX = fromX + (toX - fromX) / 2.0
        return 1 - ((x - centerX) / (centerX - fromX)).pow(2)
    }
}

class GaussianMembershipFunction(
    private val fromX: Double,
    private val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val centerX = fromX + (toX - fromX) / 2.0
        val sigma = (centerX - fromX) / 3 // (centerX - fromX) is too big
        val numerator = - (x - centerX).pow(2)
        val denominator = 2 * sigma.pow(2)
        return exp(numerator / denominator)
    }
}

fun buildEvenTerms( // distribute terms from min to max evenly
    termsCount: Int,
    minValue: Int, maxValue: Int,
    builder: MembershipFunctionBuilder
): List<MembershipFunction> {
    val step = (maxValue - minValue) / (termsCount.toDouble() - 1) * 2
    val fromX = minValue - step / 2

    return List(termsCount) { i ->
        builder(
            fromX + step / 2 * i,
            fromX + step * (i / 2 + 1)
        )
    }
}

/* Used only by following code blocks */

private val columnNameToTermNameToMembershipFunction: Map<String, Map<String, MembershipFunction>> =
    variables.map { variable ->
        val termsMemFuns: List<MembershipFunction> = buildEvenTerms(
            termsCount = variable.termNames.size,
            minValue = variable.minValue,
            maxValue = variable.maxValue,
            builder = MembershipFunctionBuilder(
                when (variable.membershipFunctionType) {
                    MembershipFunctionType.TRIANGULAR -> ::TriangularMembershipFunction
                    MembershipFunctionType.TRAPEZOIDAL -> ::TrapezoidalMembershipFunction
                    MembershipFunctionType.PARABOLIC -> ::ParabolicMembershipFunction
                    MembershipFunctionType.GAUSSIAN -> ::GaussianMembershipFunction
                }
            )
        )
        val termNameToFunction: Map<String, MembershipFunction> =
            variable.termNames
                .zip(termsMemFuns)
                .toMap()

        variable.columnName to termNameToFunction
    }.toMap()

fun getMembershipFunction(columnName: String, termName: String): MembershipFunction =
    columnNameToTermNameToMembershipFunction[columnName]!![termName]!!

operator fun LinguisticVariable.get(termName: String): MembershipFunction =
    columnNameToTermNameToMembershipFunction[columnName]!![termName]!!

Прежде чем реализовать машину нечёткого вывода Мамдани, определим набор правил:

In [5]:
import java.util.HashMap

typealias InputTerms = Map<String, String> // variable to term
data class ExpectedOutput(val confidenceDegree: Double, val termName: String)

// input terms (variable to term) to output term
private val intermediateRules: MutableMap<InputTerms, ExpectedOutput> = HashMap()

learnData.forEach { row ->
    data class TermAndValue(val termName: String, val memFunValue: Double)

    // variable to term
    val termsWithMaxMembership: Map<LinguisticVariable, TermAndValue> = variables.map { variable ->
        val value: Double = (row[variable.columnName] as Number).toDouble() // some cell are parsed as Int, some - as Double
        val termWithMaxMembership: TermAndValue = variable.termNames.asSequence()
            .map { termName ->
                TermAndValue(
                    termName = termName,
                    memFunValue = variable[termName](value)
                )
            }.maxBy(TermAndValue::memFunValue)
        variable to termWithMaxMembership
    }.toMap()

    val inputTerms: InputTerms = termsWithMaxMembership.filterKeys { variable ->
        inputVariables.contains(variable)
    }.map { (variable: LinguisticVariable, termAndValue: TermAndValue) ->
        variable.columnName to termAndValue.termName
    }.toMap()

    val confidenceDegree: Double = termsWithMaxMembership.asSequence()
        .map(Map.Entry<*, TermAndValue>::value)
        .map(TermAndValue::memFunValue)
        .reduce(Double::times)

    if (confidenceDegree > intermediateRules[inputTerms]?.confidenceDegree ?: 0.0)
        intermediateRules[inputTerms] = ExpectedOutput(
            confidenceDegree = confidenceDegree,
            termName = termsWithMaxMembership[outputVariable]!!.termName
        )
}

/* Result */

data class Rule(val inputTerms: InputTerms, val outputTerm: String) {
    override fun toString(): String =
        StringBuilder()
            .append("- ЕСЛИ ")
            .apply {
                inputTerms.asSequence()
                    .flatMap { (columnName, term) ->
                        val userFriendlyName = inputVariables
                            .first { it.columnName == columnName }
                            .userFriendlyName

                        sequenceOf("\"$userFriendlyName\" = \"$term\"", " И ")
                    }.filterIndexed { i: Int, _: String ->
                        i < inputTerms.size * 2 - 1 // drop last "И"
                    }.forEach(this@apply::append)
            }.append(", ТОГДА \"${outputVariable.userFriendlyName}\" = \"$outputTerm\"")
            .toString()
}

val rules: List<Rule> = intermediateRules.map { (input: InputTerms, output: ExpectedOutput) ->
    Rule(inputTerms = input, outputTerm = output.termName)
}.onEach(::println)

private val orderedInputOutputVars = inputVariables + outputVariable
dataFrameOf( // display rules as table
    headers = orderedInputOutputVars.map(LinguisticVariable::userFriendlyName),
    cells = rules.map { rule ->
        inputVariables.map { rule.inputTerms[it.columnName] } + rule.outputTerm
    }
)

- ЕСЛИ "Ежедневное использование соцсети (мин)" = "много" И "Лайков поставлено (в день)" = "много" И "Просмотрено reels" = "немного", ТОГДА "PSS-10" = "умеренный стресс"
- ЕСЛИ "Ежедневное использование соцсети (мин)" = "немного" И "Лайков поставлено (в день)" = "много" И "Просмотрено reels" = "немного", ТОГДА "PSS-10" = "низкий стресс"
- ЕСЛИ "Ежедневное использование соцсети (мин)" = "много" И "Лайков поставлено (в день)" = "немного" И "Просмотрено reels" = "немного", ТОГДА "PSS-10" = "низкий стресс"
- ЕСЛИ "Ежедневное использование соцсети (мин)" = "много" И "Лайков поставлено (в день)" = "много" И "Просмотрено reels" = "много", ТОГДА "PSS-10" = "умеренный стресс"
- ЕСЛИ "Ежедневное использование соцсети (мин)" = "немного" И "Лайков поставлено (в день)" = "немного" И "Просмотрено reels" = "очень много", ТОГДА "PSS-10" = "умеренный стресс"
- ЕСЛИ "Ежедневное использование соцсети (мин)" = "немного" И "Лайков поставлено (в день)" = "немного" И "Просмотрено reels" = "много", ТОГДА "PSS

Ежедневное использование соцсети (мин),Лайков поставлено (в день),Просмотрено reels,PSS-10
много,много,немного,умеренный стресс
немного,много,немного,низкий стресс
много,немного,немного,низкий стресс
много,много,много,умеренный стресс
немного,немного,очень много,умеренный стресс
немного,немного,много,низкий стресс
много,немного,очень много,низкий стресс
немного,много,очень много,низкий стресс
очень много,очень много,очень много,высокий стресс
очень много,много,много,высокий стресс


Далее, реализуем машину нечёткого вывода [Мамдани](https://docs.exponenta.ru/R2021a_nmtnew/fuzzy/types-of-fuzzy-inference-systems.html).
<br/>Алгоритм следующий:
1. Фаззификация входных значений (получить нечёткие значения).
2. Применяем операцию AND (min).
3. Применяем операцию импликации (min)
    - `A => B = not (A and not B)`.
4. Применить операцию агрегации (max).
5. Дефаззифицируем результат (ищем центроид).

Формула координаты `x` центроида (для непрерывного случая):
$$\bar{x} = \frac{\int x \cdot \mu(x) \, dx}{\int \mu(x) \, dx}$$
Используем метод прямоугольников из численного интегрирования:
$$\bar{x} = \frac{\sum_{i=1}^{n} x_i \cdot \mu(x_i)}{\sum_{i=1}^{n} \mu(x_i)}$$

In [6]:
/**
 * @param inputValues exact values (map of linguistic variable column to its value)
 */
fun fuzzifyImplicateAggregate(inputValues: Map<String, Double>, rules: List<Rule>): MembershipFunction =
    rules.asSequence().map { rule ->
        // apply fuzzy AND (min) to input terms
        val fuzzifiedInputs: Double = rule.inputTerms.minOf { (columnName, termName) ->
            getMembershipFunction(columnName = columnName, termName = termName)(inputValues[columnName]!!)
        }
        // apply implication (min)
        MembershipFunction { x ->
            min(
                fuzzifiedInputs,
                getMembershipFunction(columnName = outputVariable.columnName, termName = rule.outputTerm)(x)
            )
        }
    }.reduce { f1, f2 -> f1 fuzzyOr f2 } // apply aggregation (max)

val MembershipFunction.centroidX: Double get() {
    val steps = 1000
    val xValues = (0 until steps).map {
        outputVariable.minValue + (outputVariable.maxValue - outputVariable.minValue) / steps.toDouble() * it
    }

    val numerator = xValues.asSequence()
        .map { it * this(it) }
        .sum()
    val denominator = xValues.asSequence()
        .map { this(it) }
        .sum()

    return numerator / denominator // find integral using rectangles method
}

fun DataRow<*>.asInputValues(): Map<String, Double> =
    inputVariables.map { variable ->
        val variableValue = (this[variable.columnName] as Number).toDouble()
        variable.columnName to variableValue
    }.toMap()

/* Mamdani fuzzy output machine solution */

// for some rows aggregatedFun(x) = 0, so centroidX is NaN (row 449 e.g.)
private val aggregatedFun: MembershipFunction = fuzzifyImplicateAggregate(
    inputValues = experimentData[100].asInputValues(),
    rules = rules
)
private val centroidX = aggregatedFun.centroidX
println("Mamdani fuzzy output machine result: $centroidX")

// visualize aggregated function and centroidX
private val steps = 1000
private val xLines: List<Double> = (0 until steps).map {
    outputVariable.minValue + (outputVariable.maxValue - outputVariable.minValue) / steps.toDouble() * it
}
private val yLines = xLines.map {
    aggregatedFun(it)
}

plot {
    layout.title = "Результат агрегации и центроид X"

    x.axis.limits = outputVariable.minValue .. outputVariable.maxValue
    y.axis.limits = 0 .. 1

    line {
        x(xLines)
        y(yLines)
        color = Color.BLUE
    }

    points {
        x(listOf(centroidX))
        y(listOf(0))
        color = Color.RED
    }
}

Mamdani fuzzy output machine result: 8.060185969192581


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="w5WSd2"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Результат агрегации и центроид X"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[0.0,40.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.0,0.04,0.08,0.12,0.16,0.2,0.24,0.28,0.32,0.36,0.4,0.44,0.48,0.52,0.56,0.6,0.64,0.68,0.72,0.76,0.8,0.84,0.88,0.92,0.96,1.0,1.04,1.08,1.12,1.16,1.2,1.24,1.28,1.32,1.36,1.4000000000000001,1.44,1.48,1.52,1.56,1.6,1.6400000000000001,1.68,1.72,1.76,1.8,1.84,1.8800000000000001,1.92,1.96,2.0,2.04,2.08,2.12,2.16,2.2,2.24,2.2800000000000002,2.32,2.36,2.4,2.44,2.48,2.52,2.56,2.6,2.64,2.68,2.72,2.7600000000000002,2.8000000000000003,2.84,2.88,2.92,2.96,3.0,3.04,3.08,3.12,3.16,3.2,3.24,3.2800000000000002,3.3200000000000003,3.36,3.4,3.44,3.48,3.52,3.56,3.6,3.64,3.68,3.72,3.7600000000000002,3.8000000000000003,3.84,3.88,3.92,3.96,4.0,4.04,4.08,4.12,4.16,4.2,4.24,4.28,4.32,4.36,4.4,4.44,4.48,4.5200000000000005,4.5600000000000005,4.6000000000000005,4.64,4.68,4.72,4.76,4.8,4.84,4.88,4.92,4.96,5.0,5.04,5.08,5.12,5.16,5.2,5.24,5.28,5.32,5.36,5.4,5.44,5.48,5.5200000000000005,5.5600000000000005,5.6000000000000005,5.64,5.68,5.72,5.76,5.8,5.84,5.88,5.92,5.96,6.0,6.04,6.08,6.12,6.16,6.2,6.24,6.28,6.32,6.36,6.4,6.44,6.48,6.5200000000000005,6.5600000000000005,6.6000000000000005,6.640000000000001,6.68,6.72,6.76,6.8,6.84,6.88,6.92,6.96,7.0,7.04,7.08,7.12,7.16,7.2,7.24,7.28,7.32,7.36,7.4,7.44,7.48,7.5200000000000005,7.5600000000000005,7.6000000000000005,7.640000000000001,7.68,7.72,7.76,7.8,7.84,7.88,7.92,7.96,8.0,8.040000000000001,8.08,8.120000000000001,8.16,8.2,8.24,8.28,8.32,8.36,8.4,8.44,8.48,8.52,8.56,8.6,8.64,8.68,8.72,8.76,8.8,8.84,8.88,8.92,8.96,9.0,9.040000000000001,9.08,9.120000000000001,9.16,9.200000000000001,9.24,9.28,9.32,9.36,9.4,9.44,9.48,9.52,9.56,9.6,9.64,9.68,9.72,9.76,9.8,9.84,9.88,9.92,9.96,10.0,10.040000000000001,10.08,10.120000000000001,10.16,10.200000000000001,10.24,10.28,10.32,10.36,10.4,10.44,10.48,10.52,10.56,10.6,10.64,10.68,10.72,10.76,10.8,10.84,10.88,10.92,10.96,11.0,11.040000000000001,11.08,11.120000000000001,11.16,11.200000000000001,11.24,11.28,11.32,11.36,11.4,11.44,11.48,11.52,11.56,11.6,11.64,11.68,11.72,11.76,11.8,11.84,11.88,11.92,11.96,12.0,12.040000000000001,12.08,12.120000000000001,12.16,12.200000000000001,12.24,12.280000000000001,12.32,12.36,12.4,12.44,12.48,12.52,12.56,12.6,12.64,12.68,12.72,12.76,12.8,12.84,12.88,12.92,12.96,13.0,13.040000000000001,13.08,13.120000000000001,13.16,13.200000000000001,13.24,13.280000000000001,13.32,13.36,13.4,13.44,13.48,13.52,13.56,13.6,13.64,13.68,13.72,13.76,13.8,13.84,13.88,13.92,13.96,14.0,14.040000000000001,14.08,14.120000000000001,14.16,14.200000000000001,14.24,14.280000000000001,14.32,14.36,14.4,14.44,14.48,14.52,14.56,14.6,14.64,14.68,14.72,14.76,14.8,14.84,14.88,14.92,14.96,15.0,15.040000000000001,15.08,15.120000000000001,15.16,15.200000000000001,15.24,15.280000000000001,15.32,15.36,15.4,15.44,15.48,15.52,15.56,15.6,15.64,15.68,15.72,15.76,15.8,15.84,15.88,15.92,15.96,16.0,16.04,16.080000000000002,16.12,16.16,16.2,16.240000000000002,16.28,16.32,16.36,16.4,16.44,16.48,16.52,16.56,16.6,16.64,16.68,16.72,16.76,16.8,16.84,16.88,16.92,16.96,17.0,17.04,17.080000000000002,17.12,17.16,17.2,17.240000000000002,17.28,17.32,17.36,17.400000000000002,17.44,17.48,17.52,

Оценим точность машины, пропустив через неё экспериментальную выборку и вычислив **корень среднеквадратичной ошибки**:
$$RMSE = \sqrt{\dfrac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2}$$

> RMSE (Root Mean Squared Error — корень из среднеквадратичной ошибки) — это метрика качества регрессионных моделей, показывающая среднюю величину отклонения прогнозов от фактических значений. Она измеряется в тех же единицах, что и целевая переменная, чувствительна к крупным ошибкам и чем меньше её значение, тем точнее модель.

По сути это среднеквадратичное отклонение, только относительно остатков (residual - разница между предсказанием и истинным значением), а не среднего:
$$\sigma = \sqrt{\dfrac{1}{N}\sum_{i=1}^{N}(x_i - \bar{x})^2}$$

Средняя абсолютная ошибка:
$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Средняя абсолютная процентная ошибка:
$$MAPE = \frac{1}{n} \sum_{i=1}^{n} \left| \frac{y_i - \hat{y}_i}{y_i} \right| \times 100\%$$

Область определения PSS от 0 до 40, но деление на 0 невозможно, поэтому для подобного случая используем **взвешенную абсолютную процентную ошибку**:
$$WMAPE = \frac{\sum |y_i - \hat{y}_i|}{\sum |y_i|} \times 100\%$$
или **симметричную абсолютную процентную ошибку**:
$$sMAPE = \frac{1}{n} \sum_{i=1}^{n} \frac{|y_i - \hat{y}_i|}{(|y_i| + |\hat{y}_i|) / 2} \times 100\%$$

In [7]:
/* Run all rows through machine in parallel */

val trueToPredictedValues: List<Pair<Double, Double>> = runBlocking {
    val CHUNK_SIZE = 10_000
    val deferredTrueToPredictedValue: List<Deferred<List<Pair<Double, Double>>>> = experimentData.asSequence()
        .chunked(CHUNK_SIZE)
        .map { chunk ->
            async(context = Dispatchers.IO) {
                chunk.map { row ->
                    val trueValue = (row[outputVariable.columnName] as Number).toDouble()
                    val predictedValue = fuzzifyImplicateAggregate(
                        inputValues = row.asInputValues(),
                        rules = rules
                    ).centroidX

                    trueValue to predictedValue
                }.filterNot { // missing rules can lead to NaN centroid (aggregated function = 0)
                    it.second.isNaN() // skip such rows
                }
            }
        }.toList()

    var trueToPredictedValues = emptyList<Pair<Double, Double>>()
    for (deferredChunk in deferredTrueToPredictedValue)
        trueToPredictedValues += deferredChunk.await()

    trueToPredictedValues
}

/* Quality metrics */

val mse = trueToPredictedValues.asSequence()
    .map { (it.first - it.second).pow(2) }
    .average()

val rmse = mse.pow(0.5)

val mae = trueToPredictedValues.asSequence()
    .map { (it.first - it.second).absoluteValue }
    .average()

private val absTrueValuesSum = trueToPredictedValues.asSequence()
    .map(Pair<Double, *>::first)
    .map(Double::absoluteValue)
    .sum()
val wmape = trueToPredictedValues.asSequence()
    .map { (it.first - it.second).absoluteValue }
    .sum() / absTrueValuesSum * 100

val smape = trueToPredictedValues.asSequence()
    .map { (it.first - it.second).absoluteValue / (it.first.absoluteValue + it.second.absoluteValue) }
    .average() * 200

/* Display */

DISPLAY(dataFrameOf(
    headers = listOf("Метрика", "Значение"),
    cells = listOf(
        listOf("MSE", mse),
        listOf("RMSE", rmse),
        listOf("MAE", mae),
        listOf("WMAPE", "$wmape%"),
        listOf("sMAPE", "$smape%")
    )
))

dataFrameOf(
    headers = (inputVariables + outputVariable)
        .map(LinguisticVariable::userFriendlyName)
        + "Вывод машины"
        + "Абсолютная ошибка",
    cells = experimentData.head(20).map { row ->
        val inputValuesMap = row.asInputValues()

        val inputValues = inputValuesMap.values.toList()
        val outputValue = (row[outputVariable.columnName] as Number).toDouble()
        val machineOutput = fuzzifyImplicateAggregate(
            inputValues = inputValuesMap,
            rules = rules
        ).centroidX
        val absResidual = (outputValue - machineOutput).absoluteValue

        inputValues + outputValue + machineOutput + absResidual
    }
)

Метрика,Значение
MSE,"66,674322"
RMSE,"8,165435"
MAE,"6,487053"
WMAPE,32.485353577233006%
sMAPE,46.61634288694441%


Ежедневное использование соцсети (мин),Лайков поставлено (в день),Просмотрено reels,PSS-10,Вывод машины,Абсолютная ошибка
"311,000000","180,000000","150,000000","37,000000","30,019454","6,980546"
"335,000000","184,000000","150,000000","29,000000","30,135635","1,135635"
"72,000000","54,000000","49,000000","15,000000","8,147921","6,852079"
"112,000000","82,000000","79,000000","8,000000","9,845611","1,845611"
"89,000000","65,000000","68,000000","9,000000","9,080575","0,080575"
"399,000000","245,000000","150,000000","38,000000","31,765812","6,234188"
"336,000000","199,000000","150,000000","38,000000","30,562716","7,437284"
"76,000000","63,000000","68,000000","7,000000","9,080575","2,080575"
"282,000000","157,000000","121,000000","20,000000","31,213173","11,213173"
"114,000000","73,000000","70,000000","14,000000","9,215862","4,784138"
